# Time-based split (no leakage)

Random split vs split by time for event data.


In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rng=np.random.default_rng(9); n=3000
day=np.sort(rng.integers(0,365,n))
x1=rng.normal(0,1,n); x2=rng.normal(0,1,n)
# drift over time
logit=0.8*x1 - 0.5*x2 + 0.002*day
y=(rng.random(n)<1/(1+np.exp(-logit))).astype(int)
df=pd.DataFrame({'day':day,'x1':x1,'x2':x2,'y':y})
# random
idx=rng.permutation(n); tr,te=idx[:2250], idx[2250:]
m=LogisticRegression(max_iter=1000).fit(df.loc[tr,['x1','x2']], df.loc[tr,'y'])
print('random auc', round(roc_auc_score(df.loc[te,'y'], m.predict_proba(df.loc[te,['x1','x2']])[:,1]),3))
# time
cut=df['day'].quantile(0.75)
tr=df['day']<=cut; te=~tr
m=LogisticRegression(max_iter=1000).fit(df.loc[tr,['x1','x2']], df.loc[tr,'y'])
print('time auc', round(roc_auc_score(df.loc[te,'y'], m.predict_proba(df.loc[te,['x1','x2']])[:,1]),3))


### Leakage
Never fit scalers or encoders on the full timeline before the split.
